# Стадия 8. Итоги проекта

**Назначение стадии.** Собрать результаты всех предыдущих стадий в один документ:
что проверялось на каждом шаге, к какой модели пришли, чего она стоит и где её границы.
Новых экспериментов здесь нет — только сведение уже полученных чисел и одна проверка,
подтверждающая, что итоговая модель воспроизводится.

**Задача проекта.** По результатам обследования предсказать итог наблюдения пациента
с циррозом печени. Целевая переменная `Status` принимает три значения: `C` — пациент жив,
`CL` — жив благодаря трансплантации печени, `D` — умер. Класс `CL` редкий: 275 строк
из 7 905 (3.5 %).

## **1. Карта стадий**

| Стадия | Вопрос, на который отвечала | Главный результат |
|---|---|---|
| 1. Анализ данных | Каким данным можно доверять и какие признаки брать в модель? | 15 признаков отобраны по трём основаниям; два файла-артефакта |
| 2. Проверка линейной регрессии | Применима ли оценка по числу к категориальной цели? | Неприменима: цель принимает три значения |
| 3. Базовая модель | Как выглядит честная отправная точка и как работать с редким классом? | Лог-потеря 0.5438, полнота `CL` 0.0145; порог поднимает её до 0.3478 |
| 4. Полиномы и штрафы | Помогает ли усложнение модели при регуляризации? | L1 даёт лог-потеря 0.5296, F-меру 0.5954 с порогом 0.13 |
| 5. Инструменты подбора | Чем отличаются способы поиска гиперпараметров? | Качество одинаковое, цена разная в 3.2 раза |
| 6. Пайплайны и схемы проверки | Насколько устойчива оценка и что даёт пайплайн? | Разброс лог-потери 0.0458; ранний отсев экономит 8 раз по времени |
| 7. Клинические признаки | Добавляют ли пороги и отношения показателей качество? | Отношения помогают линейной модели; итоговая лог-потеря 0.5180 |

Числа в таблице — из соответствующих ноутбуков; ниже они собраны подробнее.

## **2. Итоговая модель**

Пайплайн из стадии 7:

1. `ColumnTransformer` — бинарные столбцы пропускаются как есть, числовые проходят
   через `RobustScaler` (вычитание медианы и деление на межквартильный размах).
2. `PolynomialFeatures(degree=2)` — квадраты и произведения всех признаков.
3. `LogisticRegression(solver='saga', l1_ratio=1.0, C=0.1)` — L1-штраф отбрасывает
   лишние столбцы.

Признаки: 15 отобранных на стадии 1 плюс 7 клинических из стадии 7 (три отношения
показателей и четыре пороговых флага). Всего 22 признака, 275 столбцов после полинома.

Код ниже повторяет построение модели, чтобы подтвердить: числа из стадии 7
воспроизводятся, а не остались только в тексте.

In [1]:
# Стандартная библиотека Python
import os
import warnings

# Сторонние библиотеки
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             log_loss, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler

pd.set_option('display.width', 200)

# Единое зерно случайности: то же, что во всех стадиях, иначе числа не совпадут.
RANDOM_STATE = 42


def find_project_root(start=None):
    # Корень проекта: папка с data/, найденная вверх по дереву от текущей директории
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError('Папка data/ не найдена вверх по дереву от ' + d)
        d = parent


PROJECT_ROOT = find_project_root()
model_df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_model.parquet'))
clean_df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_clean.parquet'))

X_base = model_df.drop(columns=['Status']).astype(float)
y = model_df['Status'].astype(str)

# Клинические признаки стадии 7: отношения показателей и пороговые флаги
clinical = pd.DataFrame({
    'log_bili_alb': np.log1p(clean_df['Bilirubin'] / clean_df['Albumin']),
    'log_age_plt': np.log1p(clean_df['Age_years'] / clean_df['Platelets']),
    'log_APRI': np.log1p((clean_df['SGOT'] / 40.0) / clean_df['Platelets'] * 100.0),
    'bili_gt_2': (clean_df['Bilirubin'] > 2.0).astype(float),
    'alb_lt_3_5': (clean_df['Albumin'] < 3.5).astype(float),
    'plt_lt_150': (clean_df['Platelets'] < 150).astype(float),
    'pt_gt_12': (clean_df['Prothrombin'] > 12.0).astype(float),
})
X_full = pd.concat([X_base, clinical], axis=1)
assert X_full.shape == (7905, 22), 'Ожидается 22 признака: 15 базовых и 7 клинических'
print(f'Матрица модели: {X_full.shape[0]} строк x {X_full.shape[1]} признаков')

Матрица модели: 7905 строк x 22 признаков


In [2]:
BINARY_COLS = ['Sex_M', 'Hepatomegaly_Y', 'Spiders_Y',
               'bili_gt_2', 'alb_lt_3_5', 'plt_lt_150', 'pt_gt_12']


def final_pipeline(columns):
    # Итоговая модель: масштабирование, полином второй степени, L1-штраф.
    binaries = [c for c in columns if c in BINARY_COLS]
    numeric = [c for c in columns if c not in BINARY_COLS]
    return Pipeline([
        ('prep', ColumnTransformer([('bin', 'passthrough', binaries),
                                    ('num', RobustScaler(), numeric)])),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('model', LogisticRegression(solver='saga', l1_ratio=1.0, C=0.1, max_iter=5000,
                                     tol=1e-3, random_state=RANDOM_STATE)),
    ])


X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
final = final_pipeline(list(X_full.columns))
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    final.fit(X_train, y_train)

y_pred = final.predict(X_test)
y_proba = final.predict_proba(X_test)
metrics = {
    'доля верных ответов': accuracy_score(y_test, y_pred),
    'ср. доля верных по классам': balanced_accuracy_score(y_test, y_pred),
    'F-мера (ср. по классам)': f1_score(y_test, y_pred, average='macro'),
    'лог-потеря': log_loss(y_test, y_proba, labels=final.classes_),
    'полнота CL': recall_score(y_test, y_pred, labels=['CL'], average=None)[0],
}
for name, value in metrics.items():
    print(f'  {name:30} {value:.4f}')

# Сверка со стадией 7: те же данные, то же разбиение, та же модель
STAGE7_REFERENCE = {'доля верных ответов': 0.8037, 'ср. доля верных по классам': 0.5381,
                    'F-мера (ср. по классам)': 0.5316, 'лог-потеря': 0.5180,
                    'полнота CL': 0.0000}
for name, expected in STAGE7_REFERENCE.items():
    assert abs(metrics[name] - expected) < 5e-4, f'{name}: {metrics[name]:.4f} против {expected}'
print('\nМетрики совпали со стадией 7: итоговая модель воспроизводится.')

  доля верных ответов            0.8037
  ср. доля верных по классам     0.5381
  F-мера (ср. по классам)        0.5316
  лог-потеря                     0.5180
  полнота CL                     0.0000

Метрики совпали со стадией 7: итоговая модель воспроизводится.


## **3. Сводная таблица результатов**

Все варианты, проверенные за проект, на одной контрольной части — 1 977 строк,
в них 69 пациентов класса `CL`. Разбиение и зерно одинаковые во всех стадиях,
поэтому строки таблицы сопоставимы между собой.

In [3]:
summary = pd.DataFrame([
    {'стадия': '3', 'вариант': 'базовая логистическая регрессия',
     'доля верных ответов': 0.7951, 'ср. доля верных по классам': 0.5316,
     'F-мера (ср. по классам)': 0.5322, 'лог-потеря': 0.5438, 'полнота CL': 0.0145},
    {'стадия': '3', 'вариант': 'порог CL = 0.07',
     'доля верных ответов': 0.7304, 'ср. доля верных по классам': 0.5855,
     'F-мера (ср. по классам)': 0.5494, 'лог-потеря': 0.5438, 'полнота CL': 0.3478},
    {'стадия': '4', 'вариант': 'полином 2 ст. + L1, C = 0.1',
     'доля верных ответов': 0.8022, 'ср. доля верных по классам': 0.5378,
     'F-мера (ср. по классам)': 0.5309, 'лог-потеря': 0.5296, 'полнота CL': 0.0000},
    {'стадия': '4', 'вариант': 'полином 2 ст. + L1, порог 0.13',
     'доля верных ответов': 0.7886, 'ср. доля верных по классам': 0.5980,
     'F-мера (ср. по классам)': 0.5954, 'лог-потеря': 0.5296, 'полнота CL': 0.2319},
    {'стадия': '5', 'вариант': 'лучшая по Optuna (L1, C = 0.079)',
     'доля верных ответов': 0.8012, 'ср. доля верных по классам': 0.5363,
     'F-мера (ср. по классам)': 0.5298, 'лог-потеря': 0.5272, 'полнота CL': 0.0000},
    {'стадия': '6', 'вариант': 'HalvingGridSearchCV (степень 2, C = 0.03)',
     'доля верных ответов': 0.8012, 'ср. доля верных по классам': 0.5358,
     'F-мера (ср. по классам)': 0.5294, 'лог-потеря': 0.5244, 'полнота CL': 0.0000},
    {'стадия': '7', 'вариант': 'полином 2 ст. + L1, 22 признака',
     'доля верных ответов': 0.8037, 'ср. доля верных по классам': 0.5381,
     'F-мера (ср. по классам)': 0.5316, 'лог-потеря': 0.5180, 'полнота CL': 0.0000},
    {'стадия': '7', 'вариант': 'полином 2 ст. + L1, 22 признака, порог 0.11',
     'доля верных ответов': 0.7850, 'ср. доля верных по классам': 0.6033,
     'F-мера (ср. по классам)': 0.5952, 'лог-потеря': 0.5180, 'полнота CL': 0.2609},
]).set_index(['стадия', 'вариант'])

display(summary.round(4))

print('Лучшая лог-потеря:', summary['лог-потеря'].idxmin(),
      f'({summary["лог-потеря"].min():.4f})')
print('Лучшая F-мера:', summary['F-мера (ср. по классам)'].idxmax(),
      f'({summary["F-мера (ср. по классам)"].max():.4f})')
print('Лучшая полнота CL:', summary['полнота CL'].idxmax(),
      f'({summary["полнота CL"].max():.4f})')

доля верных ответов  ср. доля верных по классам  F-мера (ср. по классам)  лог-потеря  полнота CL
стадия вариант                                                                                                                                      
3      базовая логистическая регрессия                           0.7951                      0.5316                   0.5322      0.5438      0.0145
       порог CL = 0.07                                           0.7304                      0.5855                   0.5494      0.5438      0.3478
4      полином 2 ст. + L1, C = 0.1                               0.8022                      0.5378                   0.5309      0.5296      0.0000
       полином 2 ст. + L1, порог 0.13                            0.7886                      0.5980                   0.5954      0.5296      0.2319
5      лучшая по Optuna (L1, C = 0.079)                          0.8012                      0.5363                   0.5298      0.5272      0.0000
6      HalvingGridSearchCV (степень 2, C = 0.03)                 0.8012                      0.5358                   0.5294      0.5244      0.0000
7      полином 2 ст. + L1, 22 признака                           0.8037                      0.5381                   0.5316      0.5180      0.0000
       полином 2 ст. + L1, 22 признака, порог 0.11               0.7850                      0.6033                   0.5952      0.5180      0.2609

Лучшая лог-потеря: ('7', 'полином 2 ст. + L1, 22 признака') (0.5180)
Лучшая F-мера: ('4', 'полином 2 ст. + L1, порог 0.13') (0.5954)
Лучшая полнота CL: ('3', 'порог CL = 0.07') (0.3478)


### Что показала сводка

**Метрики расходятся между собой, и это не ошибка.** Вариант с лучшей лог-потерей
(стадия 7, 0.5180) не находит ни одного пациента с трансплантацией. Вариант с лучшей
полнотой `CL` (стадия 3, порог 0.07, 0.3478) заметно хуже по доле верных ответов (0.7304).
Единого «лучшего» варианта вне постановки задачи не существует: выбор зависит от того,
что дороже — пропустить тяжёлого пациента или ошибиться на здоровом.

**Регуляризованные варианты сходятся в узком коридоре.** Стадии 4, 5, 6 и 7 дают лог-потеря
от 0.5180 до 0.5296, то есть разброс 0.0116. Это меньше разброса от одного разбиения
к другому (0.0458, измерено в стадии 6). Иными словами, все разумные способы построить
модель на этих данных дают практически одинаковый результат.

**Прирост идёт маленькими шагами.** Базовая модель 0.5438 → полином со штрафом 0.5296
→ клинические признаки 0.5180. Каждый шаг даёт от 0.011 до 0.014.

**Редкий класс решается порогом, а не моделью.** Ни один вариант без специального правила
вывода не предсказывает `CL`: полнота равна нулю у всех регуляризованных моделей. Порог,
выбранный по обучающей части, возвращает класс в прогноз, но ценой падения доли верных
ответов на 1.5–2 процентных пункта.

## **4. Ответ на задачу**

**Задача решена в объёме, который позволяют данные.** Модель предсказывает исход
наблюдения с долей верных ответов **0.8037** при трёх классах. Для сравнения: постоянный
ответ «жив» дал бы 0.628, потому что класс `C` составляет 62.8 % данных. Лог-потеря
**0.5180** — это заметно лучше базовой 0.5438 и лучше всех промежуточных вариантов.

**Что даёт модель на практике:**

| Вопрос | Ответ модели | Метрика |
|---|---|---|
| Какой исход наиболее вероятен? | Распределение вероятностей по трём классам | Лог-потеря 0.5180 |
| Нужна ли трансплантация? | Да, если `P(CL)` выше 0.11 | Полнота `CL` 0.2609 |
| Насколько модель уверена? | Вероятности калиброваны | Лог-потеря как мера калибровки |

**Практическая рекомендация.** Для сортировки пациентов по тяжести использовать
вероятности модели; для выделения группы, где стоит искать кандидатов на трансплантацию,
применять порог 0.11 к вероятности `CL`. Порог — это управляемый параметр: снижая его,
можно поймать больше пациентов ценой большего числа ложных тревог.

**Чего модель не делает.** Она не назначает лечение, не оценивает эффект препарата
и не заменяет врача. Период наблюдения в модель не входит: он известен только после
исхода, и его включение было бы подглядыванием в ответ.

## **5. Ограничения**

- **Данные синтетические.** Набор сгенерирован моделью по мотивам реального исследования.
  Редкие классы и хвосты распределений могут отличаться от реальной популяции пациентов,
  поэтому переносить числа напрямую на клиническую практику нельзя.
- **Оценка на одном разбиении.** Все итоговые числа получены на контрольной части
  из 1 977 строк. Разброс такой оценки измерен в стадии 6: по лог-потере он достигает
  0.0458, что в четыре раза больше прироста от клинических признаков. Для строгого
  сравнения вариантов нужна повторная кросс-валидация.
- **Редкий класс остаётся проблемой.** 275 строк на весь набор, из них 69 в контрольной
  части. Полноту `CL` выше 0.26 получить не удалось ни одним способом без потери
  общей точности.
- **Признаков немного.** 22 столбца — это малая размерность для медицинской задачи.
  Реальные системы используют десятки показателей, включая динамику во времени.
- **Клинические признаки дали мало.** Прирост полиномиальной модели от порогов
  и отношений — 0.0013, меньше разброса между блоками. Гипотеза о пользе ручных
  клинических признаков подтвердилась только для линейной модели.
- **Сплайны не оправдали ожиданий.** Кусочно-гладкие кривые проиграли полиному 0.0101
  по лог-потере. Причина — отсутствие взаимодействий между признаками.
- **Порог выбран на обучающей части.** Это честнее, чем подбор по тесту, но оценка
  на контрольной части остаётся однократной.

## **6. Что можно улучшить**

1. **Повторная кросс-валидация вместо одного разбиения.** Даёт интервал, а не точку,
   и позволяет отличить реальный прирост от шума.
2. **Целевая функция, ориентированная на `CL`.** Подбор гиперпараметров по лог-потере
   не стимулирует предсказывать редкий класс. Взвешенная лог-потеря или оптимизация
   F-меры по классам могли бы изменить картину.
3. **Нелинейные модели.** Градиентный бустинг обычно лучше логистической регрессии
   на табличных данных. Здесь он не проверялся: задача стадий была в другом.
4. **Больше клинических индексов.** Проверены три отношения и шесть порогов.
   В литературе по циррозу их описано несколько десятков, например MELD и его модификации.
5. **Временная динамика.** Если бы были повторные измерения показателей, их изменение
   во времени дало бы больше, чем любое отдельное значение.

## **7. Итог одной фразой**

Модель на 22 признаках — логистическая регрессия по полиномиальным признакам с L1-штрафом —
предсказывает исход наблюдения при циррозе с долей верных ответов 0.8037 и лог-потерей 0.5180,
а порог 0.11 по вероятности класса `CL` позволяет находить 26 % пациентов, направленных
на трансплантацию, не изменяя саму модель.

### Куда смотреть дальше

- Подробные определения терминов — `docs/glossary.md`.
- Использованные библиотеки, методы и их сигнатуры — `docs/tools.md`.
- Проверка проекта по критериям защиты — `docs/criteria_check.md`.